# ***De Novo* Design of Miniprotein Inhibitors of Bacterial Adhesins** - Implementation

Implementation of the [*De Novo* Design of Miniprotein Inhibitors of Bacterial Adhesins](https://www.biorxiv.org/content/10.1101/2025.08.18.670751v1) protein design process:

> Ten thousand backbone designs were generated per target using the target crystal structure as input (PDB codes; Abp1d: 8DEZ, Abp2d: 8DF0, FimH HAS: 1UWF, FimH LAS: 3JWN) as well as hotspots (Abp1d: residues 55/93/110, Abp2d: 55/93/110, FimH: residues 54/133/137/51 or 1/13/48/138 or 52/55/135/140/164) corresponding to the substrate binding pocket residues. 20 sequences were assigned per backbone with ProteinMPNN before AF2 filtering. Designs with pAE under 10 and pLDDT over 80 were then resampled using partial diffusion, followed by MPNN and AF2 as before. This partial diffusion process was repeated once more. Abp backbones were then filtered to confirm their proximity (residues within 5 Å of the hotspots) to these “hotspot” residues and their likelihood of forming stable monomers, assessed by secondary structure topology analysis. Final designs were selected using AF2 Initial Guess metrics for the complex (pLDDT_binder $\geq 90$, pAE_interaction $\leq 6.5$), Rosetta interface metrics (DDG $\leq -30$, SAP_score $\leq 40$), and AF2 monomer metrics for the minibinder alone (pLDDT_monomer $\geq 90$).

The notebook is based off of [better-diffusion](https://colab.research.google.com/drive/1GFgie508wgpLa3GO8eWqUpc_tGDcXCkB?usp=sharing). If you're new to **RFdiffusion**, please take a look at that notebook first, as it is much better documented and more accessible for beginners.

## Quick start (non-technical)

This notebook guides you through the *De Novo* miniprotein inhibitor workflow from the paper. You can run it top-to-bottom without editing code.

**Steps:**
1. **Run the setup cells** to install RFdiffusion, ProteinMPNN, AlphaFold, and Rosetta assets.
2. **Choose a target preset** and confirm hotspot residues.
3. **Run the design loop** (backbone generation → MPNN → AF2 → partial diffusion).
4. **Filter final designs** using hotspot proximity and optional Rosetta/monomer/topology metrics.

**Paper-scale reference:** The paper uses *10,000* backbones per target and *20* sequences per backbone. That is very slow on Colab. Start small, then scale up if needed.

**Important:** This is a research workflow. Computational filters do not guarantee real-world activity; experimental validation is still required.


In [ ]:
#@title Setup **RFdiffusion** + **ProteinMPNN** + **AlphaFold**
%%time
import os, time, signal
import sys, random, string, re
if not os.path.isdir("params"):
  os.system("apt-get install aria2")
  os.system("mkdir params")
  # send param download into background
  os.system("(\
  aria2c -q -x 16 https://files.ipd.uw.edu/krypton/schedules.zip; \
  aria2c -q -x 16 http://files.ipd.uw.edu/pub/RFdiffusion/6f5902ac237024bdd0c176cb93063dc4/Base_ckpt.pt; \
  aria2c -q -x 16 http://files.ipd.uw.edu/pub/RFdiffusion/e29311f6f1bf1af907f9ef9f44b8328b/Complex_base_ckpt.pt; \
  aria2c -q -x 16 http://files.ipd.uw.edu/pub/RFdiffusion/f572d396fae9206628714fb2ce00f72e/Complex_beta_ckpt.pt; \
  aria2c -q -x 16 https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar; \
  tar -xf alphafold_params_2022-12-06.tar -C params; \
  touch params/done.txt) &")

if not os.path.isdir("RFdiffusion"):
  print("installing RFdiffusion...")
  os.system("git clone https://github.com/sokrypton/RFdiffusion.git")
  os.system("pip install jedi omegaconf hydra-core icecream pyrsistent pynvml decorator")
  os.system("pip install git+https://github.com/NVIDIA/dllogger#egg=dllogger")
  # 17Mar2024: adding --no-dependencies to avoid installing nvidia-cuda-* dependencies
  # 25Aug2025: updating dgi install to work with latest pytorch
  os.system("pip install --no-dependencies dgl -f https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html")
  os.system("pip install --no-dependencies e3nn==0.5.5 opt_einsum_fx")
  os.system("cd RFdiffusion/env/SE3Transformer; pip install .")
  os.system("wget -qnc https://files.ipd.uw.edu/krypton/ananas")
  os.system("chmod +x ananas")

if not os.path.isdir("colabdesign"):
  print("installing ColabDesign...")
  os.system("pip -q install git+https://github.com/sokrypton/ColabDesign.git")
  os.system("ln -s /usr/local/lib/python3.*/dist-packages/colabdesign colabdesign")

if not os.path.isdir("RFdiffusion/models"):
  print("downloading RFdiffusion params...")
  os.system("mkdir RFdiffusion/models")
  models = ["Base_ckpt.pt","Complex_base_ckpt.pt","Complex_beta_ckpt.pt"]
  for m in models:
    while os.path.isfile(f"{m}.aria2"):
      time.sleep(5)
  os.system(f"mv {' '.join(models)} RFdiffusion/models")
  os.system("unzip schedules.zip; rm schedules.zip")

if 'RFdiffusion' not in sys.path:
  os.environ["DGLBACKEND"] = "pytorch"
  sys.path.append('RFdiffusion')

from google.colab import files
import json
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML
import ipywidgets as widgets
import py3Dmol

from inference.utils import parse_pdb
from colabdesign.rf.utils import get_ca
from colabdesign.rf.utils import fix_contigs, fix_partial_contigs, fix_pdb, sym_it
from colabdesign.shared.protein import pdb_to_string
from colabdesign.shared.plot import plot_pseudo_3D

def get_pdb(pdb_code=None):
  if pdb_code is None or pdb_code == "":
    upload_dict = files.upload()
    pdb_string = upload_dict[list(upload_dict.keys())[0]]
    with open("tmp.pdb","wb") as out: out.write(pdb_string)
    return "tmp.pdb"
  elif os.path.isfile(pdb_code):
    return pdb_code
  elif len(pdb_code) == 4:
    if not os.path.isfile(f"{pdb_code}.pdb1"):
      os.system(f"wget -qnc https://files.rcsb.org/download/{pdb_code}.pdb1.gz")
      os.system(f"gunzip {pdb_code}.pdb1.gz")
    return f"{pdb_code}.pdb1"
  else:
    os.system(f"wget -qnc https://alphafold.ebi.ac.uk/files/AF-{pdb_code}-F1-model_v3.pdb")
    return f"AF-{pdb_code}-F1-model_v3.pdb"

def run_ananas(pdb_str, path, sym=None):
  pdb_filename = f"outputs/{path}/ananas_input.pdb"
  out_filename = f"outputs/{path}/ananas.json"
  with open(pdb_filename,"w") as handle:
    handle.write(pdb_str)

  cmd = f"./ananas {pdb_filename} -u -j {out_filename}"
  if sym is None: os.system(cmd)
  else: os.system(f"{cmd} {sym}")

  # parse results
  try:
    out = json.loads(open(out_filename,"r").read())
    results,AU = out[0], out[-1]["AU"]
    group = AU["group"]
    chains = AU["chain names"]
    rmsd = results["Average_RMSD"]
    print(f"AnAnaS detected {group} symmetry at RMSD:{rmsd:.3}")

    C = np.array(results['transforms'][0]['CENTER'])
    A = [np.array(t["AXIS"]) for t in results['transforms']]

    # apply symmetry and filter to the asymmetric unit
    new_lines = []
    for line in pdb_str.split("\n"):
      if line.startswith("ATOM"):
        chain = line[21:22]
        if chain in chains:
          x = np.array([float(line[i:(i+8)]) for i in [30,38,46]])
          if group[0] == "c":
            x = sym_it(x,C,A[0])
          if group[0] == "d":
            x = sym_it(x,C,A[1],A[0])
          coord_str = "".join(["{:8.3f}".format(a) for a in x])
          new_lines.append(line[:30]+coord_str+line[54:])
      else:
        new_lines.append(line)
    return results, "\n".join(new_lines)

  except:
    return None, pdb_str

def run(command, steps, num_designs=1, visual="none"):

  def run_command_and_get_pid(command):
    pid_file = '/dev/shm/pid'
    os.system(f'nohup {command} & echo $! > {pid_file}')
    with open(pid_file, 'r') as f:
      pid = int(f.read().strip())
    os.remove(pid_file)
    return pid
  def is_process_running(pid):
    try:
      os.kill(pid, 0)
    except OSError:
      return False
    else:
      return True

  run_output = widgets.Output()
  progress = widgets.FloatProgress(min=0, max=1, description='running', bar_style='info')
  display(widgets.VBox([progress, run_output]))

  # clear previous run
  for n in range(steps):
    if os.path.isfile(f"/dev/shm/{n}.pdb"):
      os.remove(f"/dev/shm/{n}.pdb")

  pid = run_command_and_get_pid(command)
  try:
    fail = False
    for _ in range(num_designs):

      # for each step check if output generated
      for n in range(steps):
        wait = True
        while wait and not fail:
          time.sleep(0.1)
          if os.path.isfile(f"/dev/shm/{n}.pdb"):
            pdb_str = open(f"/dev/shm/{n}.pdb").read()
            if pdb_str[-3:] == "TER":
              wait = False
            elif not is_process_running(pid):
              fail = True
          elif not is_process_running(pid):
            fail = True

        if fail:
          progress.bar_style = 'danger'
          progress.description = "failed"
          break

        else:
          progress.value = (n+1) / steps
          if visual != "none":
            with run_output:
              run_output.clear_output(wait=True)
              if visual == "image":
                xyz, bfact = get_ca(f"/dev/shm/{n}.pdb", get_bfact=True)
                fig = plt.figure()
                fig.set_dpi(100);fig.set_figwidth(6);fig.set_figheight(6)
                ax1 = fig.add_subplot(111);ax1.set_xticks([]);ax1.set_yticks([])
                plot_pseudo_3D(xyz, c=bfact, cmin=0.5, cmax=0.9, ax=ax1)
                plt.show()
              if visual == "interactive":
                view = py3Dmol.view(js='https://3dmol.org/build/3Dmol.js')
                view.addModel(pdb_str,'pdb')
                view.setStyle({'cartoon': {'colorscheme': {'prop':'b','gradient': 'roygb','min':0.5,'max':0.9}}})
                view.zoomTo()
                view.show()
        if os.path.exists(f"/dev/shm/{n}.pdb"):
          os.remove(f"/dev/shm/{n}.pdb")
      if fail:
        progress.bar_style = 'danger'
        progress.description = "failed"
        break

    while is_process_running(pid):
      time.sleep(0.1)

  except KeyboardInterrupt:
    os.kill(pid, signal.SIGTERM)
    progress.bar_style = 'danger'
    progress.description = "stopped"

def run_diffusion(contigs, path, pdb=None, iterations=50,
                  symmetry="none", order=1, hotspot=None,
                  chains=None, add_potential=False, partial_T="auto",
                  num_designs=1, use_beta_model=False, visual="none"):

  full_path = f"outputs/{path}"
  os.makedirs(full_path, exist_ok=True)
  opts = [f"inference.output_prefix={full_path}",
          f"inference.num_designs={num_designs}"]

  if chains == "": chains = None

  # determine symmetry type
  if symmetry in ["auto","cyclic","dihedral"]:
    if symmetry == "auto":
      sym, copies = None, 1
    else:
      sym, copies = {"cyclic":(f"c{order}",order),
                     "dihedral":(f"d{order}",order*2)}[symmetry]
  else:
    symmetry = None
    sym, copies = None, 1

  # determine mode
  contigs = contigs.replace(","," ").replace(":"," ").split()
  is_fixed, is_free = False, False
  fixed_chains = []
  for contig in contigs:
    for x in contig.split("/"):
      a = x.split("-")[0]
      if a[0].isalpha():
        is_fixed = True
        if a[0] not in fixed_chains:
          fixed_chains.append(a[0])
      if a.isnumeric():
        is_free = True
  if len(contigs) == 0 or not is_free:
    mode = "partial"
  elif is_fixed:
    mode = "fixed"
  else:
    mode = "free"

  # fix input contigs
  if mode in ["partial","fixed"]:
    pdb_str = pdb_to_string(get_pdb(pdb), chains=chains)
    if symmetry == "auto":
      a, pdb_str = run_ananas(pdb_str, path)
      if a is None:
        print(f'ERROR: no symmetry detected')
        symmetry = None
        sym, copies = None, 1
      else:
        if a["group"][0] == "c":
          symmetry = "cyclic"
          sym, copies = a["group"], int(a["group"][1:])
        elif a["group"][0] == "d":
          symmetry = "dihedral"
          sym, copies = a["group"], 2 * int(a["group"][1:])
        else:
          print(f'ERROR: the detected symmetry ({a["group"]}) not currently supported')
          symmetry = None
          sym, copies = None, 1

    elif mode == "fixed":
      pdb_str = pdb_to_string(pdb_str, chains=fixed_chains)

    pdb_filename = f"{full_path}/input.pdb"
    with open(pdb_filename, "w") as handle:
      handle.write(pdb_str)

    parsed_pdb = parse_pdb(pdb_filename)
    opts.append(f"inference.input_pdb={pdb_filename}")
    if mode in ["partial"]:
      if partial_T == "auto":
        iterations = int(80 * (iterations / 200))
      else:
        iterations = int(partial_T)
      opts.append(f"diffuser.partial_T={iterations}")
      contigs = fix_partial_contigs(contigs, parsed_pdb)
    else:
      opts.append(f"diffuser.T={iterations}")
      contigs = fix_contigs(contigs, parsed_pdb)
  else:
    opts.append(f"diffuser.T={iterations}")
    parsed_pdb = None
    contigs = fix_contigs(contigs, parsed_pdb)

  if hotspot is not None and hotspot != "":
    hotspot = ",".join(hotspot.replace(","," ").split())
    opts.append(f"ppi.hotspot_res='[{hotspot}]'")

  # setup symmetry
  if sym is not None:
    sym_opts = ["--config-name symmetry", f"inference.symmetry={sym}"]
    if add_potential:
      sym_opts += ["'potentials.guiding_potentials=[\"type:olig_contacts,weight_intra:1,weight_inter:0.1\"]'",
                   "potentials.olig_intra_all=True","potentials.olig_inter_all=True",
                   "potentials.guide_scale=2","potentials.guide_decay=quadratic"]
    opts = sym_opts + opts
    contigs = sum([contigs] * copies,[])

  opts.append(f"'contigmap.contigs=[{' '.join(contigs)}]'")
  opts += ["inference.dump_pdb=True","inference.dump_pdb_path='/dev/shm'"]
  if use_beta_model:
    opts += ["inference.ckpt_override_path=./RFdiffusion/models/Complex_beta_ckpt.pt"]

  print("mode:", mode)
  print("output:", full_path)
  print("contigs:", contigs)

  opts_str = " ".join(opts)
  cmd = f"./RFdiffusion/run_inference.py {opts_str}"
  print(cmd)

  # RUN
  run(cmd, iterations, num_designs, visual=visual)

  # fix pdbs
  for n in range(num_designs):
    pdbs = [f"outputs/traj/{path}_{n}_pX0_traj.pdb",
            f"outputs/traj/{path}_{n}_Xt-1_traj.pdb",
            f"{full_path}_{n}.pdb"]
    for pdb in pdbs:
      with open(pdb,"r") as handle: pdb_str = handle.read()
      with open(pdb,"w") as handle: handle.write(fix_pdb(pdb_str, contigs))

  return contigs, copies

installing RFdiffusion...
installing ColabDesign...
downloading RFdiffusion params...
CPU times: user 10.5 s, sys: 1.83 s, total: 12.3 s
Wall time: 1min 37s


In [ ]:
#@title Setup **Rosetta**
%%bash
if [ ! -d params/tr ]; then
  mkdir -p params/tr
  wget -qnc https://files.ipd.uw.edu/krypton/TrRosetta/models.zip -P params/tr/
  wget -qnc https://files.ipd.uw.edu/krypton/TrRosetta/bkgr_models.zip -P params/tr/
  unzip -qqo params/tr/models.zip -d params/tr/
  unzip -qqo params/tr/bkgr_models.zip -d params/tr/
  rm params/tr/models.zip
  rm params/tr/bkgr_models.zip
fi

In [ ]:
%%time
#@title Run settings
#@markdown ## 1) Optional target preset (recommended for beginners)
preset = "Abp1d (8DEZ)" #@param ["Custom", "Abp1d (8DEZ)", "Abp2d (8DF0)", "FimH HAS (1UWF) - set A", "FimH HAS (1UWF) - set B", "FimH HAS (1UWF) - set C", "FimH LAS (3JWN) - set A", "FimH LAS (3JWN) - set B", "FimH LAS (3JWN) - set C"]
apply_preset = True #@param {type:"boolean"}
#@markdown When enabled, preset values overwrite the fields below when this cell runs.

#@markdown ## 2) Method-specific parameters
num_designs = 1 #@param {type:"integer"}
#@markdown The number of backbone designs to generate with RFdiffusion.
num_seqs = 4 #@param {type:"integer"}
#@markdown Number of protein sequences to generate per backbone design.
pae_threshold = 10 #@param {type:"integer"}
#@markdown Maximum AF2 pAE threshold for filtering initially generated sequences.
plddt_threshold = 0.8 #@param {type:"number"}
#@markdown Minimum AF2 pLDDT threshold (0-1 scale; 0.8 = 80).
num_sampled_designs = 8 #@param {type: "integer"}
#@markdown Number of partial diffusion samples to sample from filtered designs.
num_sampling_loops = 2 #@param {type: "integer"}
#@markdown Number of filtering + partial diffusion loops to run before final evaluation.
proximity_threshold = 5 #@param {type:"integer"}
#@markdown Minimum distance (in Å) from binder residues to each hotspot.
plddt_binder_threshold = 0.9 #@param {type:"number"}
#@markdown Minimum AF2 pLDDT for the final binder (0-1 scale).
pae_interaction_threshold = 6.5 #@param {type:"number"}
#@markdown Maximum AF2 interaction pAE for the final complex.
ddg_threshold = -30 #@param {type:"integer"}
#@markdown Minimum Rosetta DDG threshold for filtering (more negative is better).
sap_threshold = 40 #@param {type:"integer"}
#@markdown Maximum Rosetta SAP score threshold for filtering.
plddt_monomer_threshold = 0.9 #@param {type:"number"}
#@markdown Minimum AF2 pLDDT for the minibinder alone (0-1 scale).

#@markdown ## 3) Target structure details
pdb = "8DEZ" #@param {type:"string"}
#@markdown 4-digit PDB code for the target crystal structure.
hotspot = "A55,A93,A110" #@param {type:"string"}
#@markdown Comma-separated hotspots (include chain IDs, e.g., A55,A93,A110).
target_chain = "A" #@param {type:"string"}
#@markdown Chain ID for the target structure in the complex output.
#@markdown Optional: comma-separated binder chain IDs (leave blank to auto-detect).
binder_chains = "" #@param {type:"string"}

#@markdown ## 4) Other parameters
#@markdown View [better-diffusion](https://colab.research.google.com/drive/1GFgie508wgpLa3GO8eWqUpc_tGDcXCkB?usp=sharing) for more details.
name = "test" #@param {type:"string"}
contigs = "A:50" #@param {type:"string"}
iterations = 50 #@param {type:"integer"}
visual = "image" #@param ["none", "image", "interactive"]
symmetry = "none" #@param ["none", "auto", "cyclic", "dihedral"]
order = 1 #@param ["1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11", "12"] {type:"raw"}
chains = "" #@param {type:"string"}
add_potential = True #@param {type:"boolean"}
partial_T = "auto" # @param ["auto", "10", "20", "40", "60", "80"]
use_beta_model = False #@param {type:"boolean"}
mpnn_sampling_temp = 0.1 #@param {type:"number"}
rm_aa = "C" #@param {type:"string"}
use_solubleMPNN = False #@param {type:"boolean"}
num_recycles = 3 #@param ["0", "1", "2", "3", "6", "12"] {type:"raw"}
use_multimer = False #@param {type:"boolean"}

# This is False by default: to replicate the method, `initial_guess` will be turned
# off for the first two loops and on for the final evaluation step below.
initial_guess = False

presets = {
  "Abp1d (8DEZ)": {"pdb": "8DEZ", "hotspot": "A55,A93,A110", "contigs": "A:50", "target_chain": "A"},
  "Abp2d (8DF0)": {"pdb": "8DF0", "hotspot": "A55,A93,A110", "contigs": "A:50", "target_chain": "A"},
  "FimH HAS (1UWF) - set A": {"pdb": "1UWF", "hotspot": "A54,A133,A137,A51", "contigs": "A:50", "target_chain": "A"},
  "FimH HAS (1UWF) - set B": {"pdb": "1UWF", "hotspot": "A1,A13,A48,A138", "contigs": "A:50", "target_chain": "A"},
  "FimH HAS (1UWF) - set C": {"pdb": "1UWF", "hotspot": "A52,A55,A135,A140,A164", "contigs": "A:50", "target_chain": "A"},
  "FimH LAS (3JWN) - set A": {"pdb": "3JWN", "hotspot": "A54,A133,A137,A51", "contigs": "A:50", "target_chain": "A"},
  "FimH LAS (3JWN) - set B": {"pdb": "3JWN", "hotspot": "A1,A13,A48,A138", "contigs": "A:50", "target_chain": "A"},
  "FimH LAS (3JWN) - set C": {"pdb": "3JWN", "hotspot": "A52,A55,A135,A140,A164", "contigs": "A:50", "target_chain": "A"},
}

if apply_preset and preset != "Custom":
  preset_values = presets[preset]
  pdb = preset_values["pdb"]
  hotspot = preset_values["hotspot"]
  contigs = preset_values["contigs"]
  target_chain = preset_values["target_chain"]
  print(f"Preset loaded: {preset} (pdb={pdb}, hotspot={hotspot})")

# determine where to save
path = name
while os.path.exists(f"outputs/{path}_0.pdb"):
  path = name + "_" + ''.join(random.choices(string.ascii_lowercase + string.digits, k=5))

if not os.path.isfile("params/done.txt"):
  print("downloading AlphaFold params...")
  while not os.path.isfile("params/done.txt"):
    time.sleep(5)

print("Parameters are set up! Run the process below.")


In [ ]:
#@title Run the process

# Generate the backbones using the target crystal structure as input,
# as well as hotspots corresponding to the substrate binding pocket residues.
flags = {"contigs":contigs,
         "pdb":pdb,
         "order":order,
         "iterations":iterations,
         "symmetry":symmetry,
         "hotspot":hotspot,
         "path":path,
         "chains":chains,
         "add_potential":add_potential,
         "num_designs":num_designs,
         "use_beta_model":use_beta_model,
         "visual":visual,
         "partial_T":partial_T}

for k,v in flags.items():
  if isinstance(v,str):
    flags[k] = v.replace("'","").replace('"','')

init_run_contigs, init_run_copies = run_diffusion(**flags)

# Assign num_seq sequences per backbone and calculate AF2 metrics
def run_mpnn_alphafold(path: str, contigs, copies, num_seqs: int = num_seqs,
                       num_recycles: int = num_recycles, rm_aa: str = rm_aa,
                       mpnn_sampling_temp: float = mpnn_sampling_temp,
                       num_designs: int = num_designs, initial_guess: bool =
                       initial_guess, use_multimer: bool = use_multimer,
                       use_solubleMPNN: bool = use_solubleMPNN) -> None:
  contigs_str = ":".join(contigs)
  opts = [f"--pdb=outputs/{path}_0.pdb",
          f"--loc=outputs/{path}",
          f"--contig={contigs_str}",
          f"--copies={copies}",
          f"--num_seqs={num_seqs}",
          f"--num_recycles={num_recycles}",
          f"--rm_aa={rm_aa}",
          f"--mpnn_sampling_temp={mpnn_sampling_temp}",
          f"--num_designs={num_designs}"]
  if initial_guess: opts.append("--initial_guess")
  if use_multimer: opts.append("--use_multimer")
  if use_solubleMPNN: opts.append("--use_soluble")
  opts = ' '.join(opts)
  !python colabdesign/rf/designability_test.py {opts}

import pandas as pd

def filter_and_resample(path: str) -> tuple[list[str], list[str], list[int]]:
  if not os.path.exists(f"./outputs/{path}/all_pdb/"):
    raise FileNotFoundError(f"There are no results for run {path}!")

  resampled_paths: list[str] = []
  resampled_contigs: list[str] = []
  resampled_copies: list[int] = []

  # Filter the generated sequences according to pae_threshold and plddt_threshold
  df: pd.DataFrame = pd.read_csv(f"outputs/{path}/mpnn_results.csv")
  df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
  if 'i_pae' not in df.columns and 'pae' in df.columns:
    df['i_pae'] = df['pae']
  if 'i_pae' not in df.columns:
    raise ValueError('No interaction pAE column found in mpnn_results.csv.')
  if 'plddt' not in df.columns:
    raise ValueError('No pLDDT column found in mpnn_results.csv.')
  filtered_df: pd.DataFrame = df[
    (df['plddt'] >= plddt_threshold) & (df['i_pae'] <= pae_threshold)
  ]

  filtered_designs: list[str] = [
    f"{path}_design{int(row['design'])}_n{int(row['n'])}"
    for _, row in filtered_df.iterrows()
  ]
  print(f"Found {len(filtered_designs)} designs from run {path} passing the"
        f"thresholds (pLDDT >= {plddt_threshold}, pAE <= {pae_threshold}).")

  if not filtered_designs:
    return resampled_paths, resampled_contigs, resampled_copies

  # Resample designs using partial diffusion
  for design in filtered_designs:
    resample_path: str = design
    while os.path.exists(f"./outputs/{resample_path}_0.pdb"):
      resample_path = name + "_" + ''.join(random.choices(string.ascii_lowercase + string.digits, k=5))

    pdb_to_resample: str = f"outputs/{path}/all_pdb/{design[len(path) + 1:]}.pdb"

    resample_flags: dict = {
      "contigs": "",
      "pdb": pdb_to_resample,
      "order":order,
      "iterations":iterations,
      "symmetry":symmetry,
      "hotspot":hotspot,
      "path": resample_path,
      "chains":chains,
      "add_potential":add_potential,
      "num_designs":num_sampled_designs,
      "use_beta_model":use_beta_model,
      "visual":visual,
      "partial_T":partial_T
    }

    for k,v in resample_flags.items():
      if isinstance(v,str):
        resample_flags[k] = v.replace("'","").replace('"','')

    contigs, copies = run_diffusion(**resample_flags)

    resampled_paths.append(resample_path)
    resampled_contigs.append(contigs)
    resampled_copies.append(copies)

  return resampled_paths, resampled_contigs, resampled_copies

curr_paths: list[str] = [path]
curr_contigs: list[str] = [init_run_contigs]
curr_copies: list[int] = [init_run_copies]

paths_to_resample: list[str] = []
contigs_to_resample: list[str] = []
copies_to_resample: list[int] = []

for loop_id in range(num_sampling_loops):
  print(f"\n--- Sampling loop {loop_id + 1} / {num_sampling_loops} ---")
  for i in range(len(curr_paths)):
    run_mpnn_alphafold(curr_paths[i], curr_contigs[i], curr_copies[i])
    resampled_paths, resampled_contigs, resampled_copies = filter_and_resample(curr_paths[i])
    paths_to_resample.extend(resampled_paths)
    contigs_to_resample.extend(resampled_contigs)
    copies_to_resample.extend(resampled_copies)
  if len(paths_to_resample) == 0:
    print("No designs passed the thresholds. Stopping early.")
    if loop_id == 0:
      curr_paths = []
    break
  curr_paths = paths_to_resample
  curr_contigs = contigs_to_resample
  curr_copies = copies_to_resample
  paths_to_resample = []
  contigs_to_resample = []
  copies_to_resample = []

final_paths = curr_paths
final_contigs = curr_contigs
final_copies = curr_copies

if len(final_paths) == 0:
  raise ValueError("No designs available for final evaluation. Adjust thresholds or increase sampling.")

print("\n--- Final evaluation with AF2 initial guess ---")
for i in range(len(final_paths)):
  run_mpnn_alphafold(final_paths[i], final_contigs[i], final_copies[i], initial_guess=True)


## Final filtering and selection

This step reproduces the paper’s final filters. Hotspot proximity is computed directly from the generated complex structures.

**Optional inputs (CSV files):**
- **Rosetta interface metrics** with columns: `design`, `n`, `ddg`, `sap_score`
- **AF2 monomer metrics** with columns: `design`, `n`, `plddt_monomer`
- **Secondary-structure topology** with columns: `design`, `n`, `topology_pass` (True/False)

If you don’t have these yet, you can leave the paths blank and run a partial filter.


In [ ]:
#@title Final filtering and selection (hotspots + optional metrics)
#@markdown Provide CSV paths if you have extra metrics. Leave blank to skip.
rosetta_metrics_csv = "" #@param {type:"string"}
monomer_metrics_csv = "" #@param {type:"string"}
topology_metrics_csv = "" #@param {type:"string"}
require_rosetta_metrics = False #@param {type:"boolean"}
require_monomer_metrics = False #@param {type:"boolean"}
require_topology_metrics = False #@param {type:"boolean"}
enable_hotspot_filter = True #@param {type:"boolean"}

import os
import re
import numpy as np
import pandas as pd
from IPython.display import display

def load_metrics(path: str, required_columns: list[str], label: str) -> pd.DataFrame | None:
  if path is None or path == "":
    return None
  if not os.path.exists(path):
    raise FileNotFoundError(f"{label} file not found: {path}")
  df = pd.read_csv(path)
  missing = [c for c in required_columns if c not in df.columns]
  if missing:
    raise ValueError(f"{label} file missing columns: {missing}")
  return df

def collect_mpnn_results(paths: list[str]) -> pd.DataFrame:
  rows = []
  for p in paths:
    csv_path = f"outputs/{p}/mpnn_results.csv"
    if not os.path.exists(csv_path):
      print(f"Skipping {p}: no mpnn_results.csv")
      continue
    df = pd.read_csv(csv_path)
    df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
    df["path"] = p
    rows.append(df)
  if not rows:
    return pd.DataFrame()
  return pd.concat(rows, ignore_index=True)

def parse_hotspots(hotspot_string: str, default_chain: str) -> list[tuple[str, int]]:
  if hotspot_string is None or hotspot_string.strip() == "":
    return []
  tokens = re.split(r"[\s,]+", hotspot_string.strip())
  parsed = []
  for token in tokens:
    if not token:
      continue
    match = re.match(r"([A-Za-z]?)(\d+)", token)
    if not match:
      raise ValueError(f"Could not parse hotspot token: {token}")
    chain = match.group(1) if match.group(1) else default_chain
    parsed.append((chain, int(match.group(2))))
  return parsed

def read_ca_coords(pdb_path: str) -> dict[str, dict[int, np.ndarray]]:
  coords: dict[str, dict[int, np.ndarray]] = {}
  with open(pdb_path, 'r') as handle:
    for line in handle:
      if not line.startswith('ATOM'):
        continue
      if line[12:16].strip() != 'CA':
        continue
      chain = line[21].strip() or '?'
      resnum = int(line[22:26].strip())
      x = float(line[30:38])
      y = float(line[38:46])
      z = float(line[46:54])
      coords.setdefault(chain, {})[resnum] = np.array([x, y, z])
  return coords

def hotspot_proximity(pdb_path: str, hotspots: list[tuple[str, int]], target_chain: str, threshold: float, binder_chain_list: list[str] | None) -> tuple[bool, float | None]:
  if not os.path.exists(pdb_path):
    return False, None
  coords = read_ca_coords(pdb_path)
  if target_chain not in coords:
    return False, None
  if binder_chain_list:
    binder_chains = binder_chain_list
  else:
    binder_chains = [c for c in coords.keys() if c != target_chain]
  if not binder_chains:
    return False, None
  binder_points = np.array([coord for c in binder_chains for coord in coords[c].values()])
  if binder_points.size == 0:
    return False, None
  distances = []
  for chain, resnum in hotspots:
    chain_coords = coords.get(chain, {})
    if resnum not in chain_coords:
      return False, None
    hotspot_coord = chain_coords[resnum]
    min_dist = float(np.min(np.linalg.norm(binder_points - hotspot_coord, axis=1)))
    distances.append(min_dist)
  if not distances:
    return False, None
  return all(d <= threshold for d in distances), float(max(distances))

paths_to_use = []
if 'final_paths' in globals():
  paths_to_use = final_paths
elif 'curr_paths' in globals():
  paths_to_use = curr_paths

if not paths_to_use:
  raise ValueError('No design paths found. Run the process cell first.')

results_df = collect_mpnn_results(paths_to_use)
if results_df.empty:
  raise ValueError('No MPNN results found. Ensure the designability step completed.')

results_df['design'] = results_df['design'].astype(int)
results_df['n'] = results_df['n'].astype(int)
results_df['design_key'] = results_df.apply(lambda r: f"design{int(r['design'])}_n{int(r['n'])}", axis=1)
results_df['pdb_path'] = results_df.apply(lambda r: f"outputs/{r['path']}/all_pdb/{r['design_key']}.pdb", axis=1)

if 'i_pae' in results_df.columns:
  results_df['pae_interaction'] = results_df['i_pae']
elif 'pae' in results_df.columns:
  results_df['pae_interaction'] = results_df['pae']
else:
  raise ValueError('No interaction pAE column found in mpnn_results.csv.')

if 'plddt' not in results_df.columns:
  raise ValueError('No pLDDT column found in mpnn_results.csv.')

results_df['pass_initial_guess'] = (
  (results_df['plddt'] >= plddt_binder_threshold) &
  (results_df['pae_interaction'] <= pae_interaction_threshold)
)

hotspots = parse_hotspots(hotspot, target_chain)
binder_chain_list = None
if 'binder_chains' in globals() and binder_chains.strip():
  binder_chain_list = [c.strip() for c in re.split(r"[\s,]+", binder_chains) if c.strip()]
if enable_hotspot_filter and hotspots:
  hotspot_pass = []
  hotspot_max_dist = []
  for _, row in results_df.iterrows():
    passed, max_dist = hotspot_proximity(row['pdb_path'], hotspots, target_chain, proximity_threshold, binder_chain_list)
    hotspot_pass.append(passed)
    hotspot_max_dist.append(max_dist if max_dist is not None else np.nan)
  results_df['pass_hotspot'] = hotspot_pass
  results_df['hotspot_max_dist'] = hotspot_max_dist
else:
  results_df['pass_hotspot'] = True
  results_df['hotspot_max_dist'] = np.nan

rosetta_df = load_metrics(rosetta_metrics_csv, ['design', 'n', 'ddg', 'sap_score'], 'Rosetta')
if rosetta_df is None:
  if require_rosetta_metrics:
    raise ValueError('Rosetta metrics required but no file provided.')
  results_df['pass_rosetta'] = True
else:
  rosetta_df['design'] = rosetta_df['design'].astype(int)
  rosetta_df['n'] = rosetta_df['n'].astype(int)
  results_df = results_df.merge(rosetta_df, on=['design', 'n'], how='left')
  results_df['pass_rosetta'] = (
    (results_df['ddg'] <= ddg_threshold) &
    (results_df['sap_score'] <= sap_threshold)
  )

monomer_df = load_metrics(monomer_metrics_csv, ['design', 'n', 'plddt_monomer'], 'Monomer AF2')
if monomer_df is None:
  if require_monomer_metrics:
    raise ValueError('Monomer metrics required but no file provided.')
  results_df['pass_monomer'] = True
else:
  monomer_df['design'] = monomer_df['design'].astype(int)
  monomer_df['n'] = monomer_df['n'].astype(int)
  results_df = results_df.merge(monomer_df, on=['design', 'n'], how='left')
  results_df['pass_monomer'] = results_df['plddt_monomer'] >= plddt_monomer_threshold

topology_df = load_metrics(topology_metrics_csv, ['design', 'n', 'topology_pass'], 'Topology')
if topology_df is None:
  if require_topology_metrics:
    raise ValueError('Topology metrics required but no file provided.')
  results_df['pass_topology'] = True
else:
  topology_df['design'] = topology_df['design'].astype(int)
  topology_df['n'] = topology_df['n'].astype(int)
  results_df = results_df.merge(topology_df, on=['design', 'n'], how='left')
  results_df['pass_topology'] = results_df['topology_pass'].fillna(False).astype(bool)

results_df['pass_final'] = (
  results_df['pass_initial_guess'] &
  results_df['pass_hotspot'] &
  results_df['pass_rosetta'] &
  results_df['pass_monomer'] &
  results_df['pass_topology']
)

print(f"Total designs evaluated: {len(results_df)}")
print(f"Final designs passing all enabled filters: {results_df['pass_final'].sum()}")

final_hits = results_df[results_df['pass_final']].copy()
final_hits = final_hits.sort_values(['plddt', 'pae_interaction'], ascending=[False, True])
display(final_hits.head(20))

os.makedirs('outputs', exist_ok=True)
results_df.to_csv(f"outputs/{name}_all_metrics.csv", index=False)
final_hits.to_csv(f"outputs/{name}_final_hits.csv", index=False)
print(f"Saved full metrics to outputs/{name}_all_metrics.csv")
print(f"Saved final shortlist to outputs/{name}_final_hits.csv")


In [ ]:
# @title Visualize AlphaFold results
import os
import re
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# Scrape the output file for the stats
data = []
try:
    # Read the explicit designability log file
    file_path = f"outputs/{path}/designability.txt"
    if os.path.exists(file_path):
        with open(file_path, 'r') as file:
            content = file.read()
            # Search for the metrics output format including the sequence
            matches = re.findall(r'design:(\d+)\s+n:(\d+)\s+mpnn:([\d.]+)\s+plddt:([\d.]+)\s+ptm:([\d.]+)\s+pae:([\d.]+)\s+rmsd:([\d.]+)\s+([A-Z]+)', content)
            for m in matches:
                data.append({
                    'design': int(m[0]),
                    'n': int(m[1]),
                    'mpnn': float(m[2]),
                    'plddt': float(m[3]),
                    'ptm': float(m[4]),
                    'pae': float(m[5]),
                    'rmsd': float(m[6]),
                    'sequence': m[7]
                })
    else:
        pass
except NameError:
    print("Run the cells above to define the path and generate outputs first!")

if not data:
    print("No data found. Please ensure the ProteinMPNN and AlphaFold block ran successfully and saved to designability.txt.")
else:
    # Pre-calculate rankings
    # lower is better for rmsd, pae, mpnn. higher is better for plddt, ptm
    metrics_directions = {'rmsd': 1, 'pae': 1, 'mpnn': 1, 'plddt': -1, 'ptm': -1}

    for d in data:
        design_data = [x for x in data if x['design'] == d['design']]
        d['ranks_overall'] = {}
        d['ranks_design'] = {}

        for metric, dir in metrics_directions.items():
            overall_sorted = sorted([x[metric] for x in data], reverse=(dir == -1))
            design_sorted = sorted([x[metric] for x in design_data], reverse=(dir == -1))
            d['ranks_overall'][metric] = overall_sorted.index(d[metric]) + 1
            d['ranks_design'][metric] = design_sorted.index(d[metric]) + 1

    # Get unique designs for the dropdown
    unique_designs = sorted(list(set([d['design'] for d in data])))
    dropdown_options = ['All Designs'] + [f'Design {d}' for d in unique_designs]

    design_dropdown = widgets.Dropdown(
        options=dropdown_options,
        value='All Designs',
        description='Filter Plots by:',
        style={'description_width': 'initial'}
    )

    # Sequence dropdown options sorted by RMSD
    data_sorted_by_rmsd = sorted(data, key=lambda x: x['rmsd'])
    seq_options = [('None (Hide)', None)] + [(f"Design {d['design']}, Seq {d['n']} (RMSD: {d['rmsd']:.3f})", d) for d in data_sorted_by_rmsd]

    sequence_dropdown = widgets.Dropdown(
        options=seq_options,
        value=None,
        description='Inspect Sequence:',
        style={'description_width': 'initial'}
    )

    plot_output = widgets.Output()
    seq_output = widgets.Output()

    def update_plots(change=None):
        with plot_output:
            clear_output(wait=True)
            selection = design_dropdown.value

            if selection == 'All Designs':
                filtered_data = data
                title_suffix = "(All Designs)"
            else:
                design_num = int(selection.split(' ')[1])
                filtered_data = [d for d in data if d['design'] == design_num]
                title_suffix = f"(Design {design_num})"

            if not filtered_data:
                print("No data for this selection.")
                return

            metrics = ['mpnn', 'plddt', 'ptm', 'pae', 'rmsd']
            metric_names = ['ProteinMPNN Score', 'pLDDT', 'pTM', 'PAE', 'RMSD']

            # Print statistics
            print(f"\n--- Statistics {title_suffix} ---")
            for m, name in zip(metrics, metric_names):
                vals = [d[m] for d in filtered_data]
                print(f"{name:18}: Mean={np.mean(vals):.3f} | Median={np.median(vals):.3f} | Min={np.min(vals):.3f} | Max={np.max(vals):.3f}")

            # Plotting
            fig, axes = plt.subplots(1, 5, figsize=(20, 4))
            fig.suptitle(f"Metrics Distributions {title_suffix}", fontsize=16)

            for i, (m, name) in enumerate(zip(metrics, metric_names)):
                vals = [d[m] for d in filtered_data]
                axes[i].hist(vals, bins=10, color='skyblue', edgecolor='black')
                axes[i].set_title(name)
                axes[i].set_xlabel("Value")
                axes[i].set_ylabel("Frequency")

            plt.tight_layout()
            plt.subplots_adjust(top=0.85)
            plt.show()

    def update_seq_info(change=None):
        with seq_output:
            clear_output(wait=True)
            d = sequence_dropdown.value
            if d is None:
                clear_output(wait=False)
                return

            total_overall = len(data)
            total_design = len([x for x in data if x['design'] == d['design']])

            html_content = f"""
            <div style="padding: 10px; border: 1px solid #ccc; border-radius: 5px; background-color: #f9f9f9; color: #333;">
                <h3 style="margin-top: 0; color: black;">Design {d['design']} - Sequence {d['n']}</h3>
                <p><strong>Amino Acid Sequence:</strong><br><span style="font-family: monospace; word-break: break-all;">{d['sequence']}</span></p>
                <table style="width: 100%; text-align: left; border-collapse: collapse;">
                    <tr style="border-bottom: 1px solid #ddd;">
                        <th style="padding: 5px;">Metric</th>
                        <th style="padding: 5px;">Value</th>
                        <th style="padding: 5px;">Rank (Overall)</th>
                        <th style="padding: 5px;">Rank (in Design {d['design']})</th>
                    </tr>
                    <tr><td style="padding: 5px;">RMSD</td><td style="padding: 5px;">{d['rmsd']}</td><td style="padding: 5px;">{d['ranks_overall']['rmsd']} / {total_overall}</td><td style="padding: 5px;">{d['ranks_design']['rmsd']} / {total_design}</td></tr>
                    <tr><td style="padding: 5px;">PAE</td><td style="padding: 5px;">{d['pae']}</td><td style="padding: 5px;">{d['ranks_overall']['pae']} / {total_overall}</td><td style="padding: 5px;">{d['ranks_design']['pae']} / {total_design}</td></tr>
                    <tr><td style="padding: 5px;">ProteinMPNN</td><td style="padding: 5px;">{d['mpnn']}</td><td style="padding: 5px;">{d['ranks_overall']['mpnn']} / {total_overall}</td><td style="padding: 5px;">{d['ranks_design']['mpnn']} / {total_design}</td></tr>
                    <tr><td style="padding: 5px;">pLDDT</td><td style="padding: 5px;">{d['plddt']}</td><td style="padding: 5px;">{d['ranks_overall']['plddt']} / {total_overall}</td><td style="padding: 5px;">{d['ranks_design']['plddt']} / {total_design}</td></tr>
                    <tr><td style="padding: 5px;">pTM</td><td style="padding: 5px;">{d['ptm']}</td><td style="padding: 5px;">{d['ranks_overall']['ptm']} / {total_overall}</td><td style="padding: 5px;">{d['ranks_design']['ptm']} / {total_design}</td></tr>
                </table>
            </div>
            """
            display(HTML(html_content))

    design_dropdown.observe(update_plots, names='value')
    sequence_dropdown.observe(update_seq_info, names='value')

    ui = widgets.VBox([
        widgets.HBox([design_dropdown, sequence_dropdown]),
        seq_output,
        plot_output
    ])
    display(ui)

    # Trigger initial plots
    update_plots(None)
    update_seq_info(None)


In [ ]:
#@title Display best result
import py3Dmol
import ipywidgets as widgets
from IPython.display import display, HTML

def plot_pdb(num = "best"):
  d = None
  actual_num = 0

  # Look up the data directly instead of parsing the file header
  if 'data' in globals() and data:
    if num == "best":
      d = sorted(data, key=lambda x: x['rmsd'])[0]
      actual_num = d['design']
    else:
      actual_num = int(num)
      design_data = [x for x in data if x['design'] == actual_num]
      if design_data:
        d = sorted(design_data, key=lambda x: x['rmsd'])[0]
  else:
    # Fallback if data isn't loaded
    if num == "best":
      with open(f"outputs/{path}/best.pdb","r") as f:
        info_line = f.readline().strip('\n')
        info = info_line.split()
        if len(info) > 3:
          actual_num = info[3]
    else:
      actual_num = num

  if d:
      total_overall = len(data)
      total_design = len([x for x in data if x['design'] == d['design']])
      title_prefix = "Overall Best:" if num == "best" else f"Best of Design {num}:"
      html_content = f"""
      <div style="padding: 10px; border: 1px solid #ccc; border-radius: 5px; background-color: #f9f9f9; color: #333; margin-bottom: 10px;">
          <h3 style="margin-top: 0; color: black;">{title_prefix} Design {d['design']} - Sequence {d['n']}</h3>
          <p><strong>Amino Acid Sequence:</strong><br><span style="font-family: monospace; word-break: break-all;">{d['sequence']}</span></p>
          <table style="width: 100%; text-align: left; border-collapse: collapse;">
              <tr style="border-bottom: 1px solid #ddd;">
                  <th style="padding: 5px;">Metric</th>
                  <th style="padding: 5px;">Value</th>
                  <th style="padding: 5px;">Rank (Overall)</th>
                  <th style="padding: 5px;">Rank (in Design {d['design']})</th>
              </tr>
              <tr><td style="padding: 5px;">RMSD</td><td style="padding: 5px;">{d['rmsd']}</td><td style="padding: 5px;">{d['ranks_overall']['rmsd']} / {total_overall}</td><td style="padding: 5px;">{d['ranks_design']['rmsd']} / {total_design}</td></tr>
              <tr><td style="padding: 5px;">PAE</td><td style="padding: 5px;">{d['pae']}</td><td style="padding: 5px;">{d['ranks_overall']['pae']} / {total_overall}</td><td style="padding: 5px;">{d['ranks_design']['pae']} / {total_design}</td></tr>
              <tr><td style="padding: 5px;">ProteinMPNN</td><td style="padding: 5px;">{d['mpnn']}</td><td style="padding: 5px;">{d['ranks_overall']['mpnn']} / {total_overall}</td><td style="padding: 5px;">{d['ranks_design']['mpnn']} / {total_design}</td></tr>
              <tr><td style="padding: 5px;">pLDDT</td><td style="padding: 5px;">{d['plddt']}</td><td style="padding: 5px;">{d['ranks_overall']['plddt']} / {total_overall}</td><td style="padding: 5px;">{d['ranks_design']['plddt']} / {total_design}</td></tr>
              <tr><td style="padding: 5px;">pTM</td><td style="padding: 5px;">{d['ptm']}</td><td style="padding: 5px;">{d['ranks_overall']['ptm']} / {total_overall}</td><td style="padding: 5px;">{d['ranks_design']['ptm']} / {total_design}</td></tr>
          </table>
      </div>
      """
      display(HTML(html_content))
  else:
      print(f"Showing result for {num}")

  hbondCutoff = 4.0
  view = py3Dmol.view(js='https://3dmol.org/build/3Dmol.js')
  pdb_str = open(f"outputs/{path}_{actual_num}.pdb",'r').read()
  view.addModel(pdb_str,'pdb',{'hbondCutoff':hbondCutoff})
  pdb_str = open(f"outputs/{path}/best_design{actual_num}.pdb",'r').read()
  view.addModel(pdb_str,'pdb',{'hbondCutoff':hbondCutoff})

  view.setStyle({"model":0},{'cartoon':{}})
  view.setStyle({"model":1},{'cartoon':{'colorscheme': {'prop':'b','gradient': 'roygb','min':0,'max':100}}})
  view.zoomTo()
  view.show()

if num_designs > 1:
  def on_change(change):
    if change['name'] == 'value':
      with output:
        output.clear_output(wait=True)
        plot_pdb(change['new'])
  dropdown = widgets.Dropdown(
    options=["best"] + [str(k) for k in range(num_designs)],
    value="best",
    description='design:',
  )
  dropdown.observe(on_change)
  output = widgets.Output()
  display(widgets.VBox([dropdown, output]))
  with output:
    plot_pdb(dropdown.value)
else:
  plot_pdb()


In [ ]:
#@title Package and download results
#@markdown If you are having issues downloading the result archive,
#@markdown try disabling your adblocker and run this cell again.
#@markdown  If that fails click on the little folder icon to the
#@markdown  left, navigate to file: `name.result.zip`,
#@markdown  right-click and select \"Download\"
#@markdown (see [screenshot](https://pbs.twimg.com/media/E6wRW2lWUAEOuoe?format=jpg&name=small)).
!zip -r {path}.result.zip outputs/{path}* outputs/traj/{path}*
files.download(f"{path}.result.zip")